# TP — Traitement automatique du langage naturel (NLP) avec NLTK et Pandas

**Objectif :** réaliser les opérations de prétraitement de textes, correction orthographique et exploration de WordNet / du corpus *State of the Union*.

Le notebook est organisé selon l'énoncé fourni :
- **Partie I — Exercice 1 :** prétraitement d'un dataset Spam/Ham.
- **Partie I — Exercice 2 :** correction orthographique d'un paragraphe.
- **Partie II — Exercice 1 :** WordNet.
- **Partie II — Exercice 2 :** corpus *State of the Union*, fréquences, concordances et collocations.

> Le fichier CSV fourni est utilisé automatiquement par le notebook. Les ressources NLTK nécessaires sont téléchargées au besoin.


## Préparation — Imports et configuration

In [ ]:
import re
import string
import collections
import pandas as pd
import numpy as np
import nltk
from pathlib import Path

from nltk.corpus import stopwords, wordnet as wn, state_union
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.collocations import BigramCollocationFinder, TrigramCollocationFinder, QuadgramCollocationFinder
from nltk.metrics import BigramAssocMeasures, TrigramAssocMeasures, QuadgramAssocMeasures

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

# Le fichier est celui fourni avec le TP.
CSV_PATH = Path("spam_ham_dataset - spam_ham_dataset.csv")

# Fallback utile si le notebook est déplacé dans un autre dossier :
if not CSV_PATH.exists():
    candidates = list(Path(".").glob("*spam*ham*.csv"))
    if candidates:
        CSV_PATH = candidates[0]

print("Fichier CSV utilisé :", CSV_PATH.resolve())


In [ ]:
# Téléchargement des ressources NLTK nécessaires.
# Certaines versions récentes de NLTK utilisent aussi punkt_tab.
resources = [
    ("corpora/stopwords", "stopwords"),
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("corpora/state_union", "state_union"),
    ("corpora/wordnet", "wordnet"),
    ("corpora/omw-1.4", "omw-1.4"),
]

for resource_path, package in resources:
    try:
        nltk.data.find(resource_path)
        print(f"✓ {package} déjà disponible")
    except LookupError:
        print(f"⬇ Téléchargement de {package} ...")
        nltk.download(package)

print("Ressources NLTK prêtes.")


# Partie I — Exercice 1 : Prétraitement du dataset Spam/Ham

### 1. Charger le jeu de données et l'afficher

In [ ]:
df = pd.read_csv(CSV_PATH)

print("Dimensions :", df.shape)
print("Colonnes :", list(df.columns))
display(df.head())


### 2. Afficher la colonne `text` et conserver uniquement `text` et `label`

In [ ]:
print("Exemples de la colonne text :")
display(df["text"].head(10))

# Conservation des deux colonnes demandées.
df = df[["text", "label"]].copy()
display(df.head())


### 3. Afficher la forme des données et la distribution de la variable dépendante

In [ ]:
print("Forme des données :", df.shape)
print("\nDistribution des labels :")
print(df["label"].value_counts(dropna=False))

print("\nDistribution en pourcentage :")
print((df["label"].value_counts(normalize=True, dropna=False) * 100).round(2).astype(str) + " %")

# Visualisation simple.
ax = df["label"].value_counts().plot(kind="bar", title="Distribution Spam / Ham", xlabel="Label", ylabel="Nombre")
ax.figure.tight_layout()


### 4. Supprimer la ponctuation avec `string`

La fonction ci-dessous utilise `string.punctuation` et retire chaque caractère de ponctuation du texte.


In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans("", "", string.punctuation))

phrase_test = "->la ponctuation, est : enlevée_de cette ? phrase.!"
print("Avant :", phrase_test)
print("Après :", remove_punctuation(phrase_test))


### 5. Ajouter la colonne `text_p` sans ponctuation

In [ ]:
df["text_p"] = df["text"].fillna("").astype(str).apply(remove_punctuation)
display(df[["text", "text_p"]].head())


### 6. Ajouter la colonne `text_lower` en minuscules

In [ ]:
df["text_lower"] = df["text_p"].str.lower()
display(df[["text_p", "text_lower"]].head())


### 7. Tokenisation avec une expression régulière (`re`)

In [ ]:
def tokenize_text(text):
    # \w+ récupère les séquences alphanumériques ; les apostrophes ayant déjà
    # été supprimées à l'étape de ponctuation, elles ne posent pas de problème.
    return re.findall(r"\b\w+\b", str(text))

df["text_tokenise"] = df["text_lower"].apply(tokenize_text)
display(df[["text_lower", "text_tokenise"]].head())


### 8. Afficher les 10 premiers stopwords anglais de NLTK

In [ ]:
english_stopwords = stopwords.words("english")
print(english_stopwords[:10])


### 9. Supprimer les stopwords et ajouter `no_stopwords`

In [ ]:
def remove_stopwords(tokens):
    return [word for word in tokens if word not in english_stopwords]

df["no_stopwords"] = df["text_tokenise"].apply(remove_stopwords)
display(df[["text_tokenise", "no_stopwords"]].head())


### 10. Stemming et lemmatisation

- **Stemming** : réduit les mots à une forme racine approximative avec `PorterStemmer`.
- **Lemmatisation** : cherche une forme lexicale (lemme) avec `WordNetLemmatizer`.


In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stem_tokens(tokens):
    return [stemmer.stem(word) for word in tokens]

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

df["text_stem"] = df["no_stopwords"].apply(stem_tokens)
df["text_lem"] = df["no_stopwords"].apply(lemmatize_tokens)

display(df[["no_stopwords", "text_stem", "text_lem"]].head(10))


### Comparaison détaillée sur quelques e-mails

In [ ]:
for i in range(min(5, len(df))):
    print(f"--- Exemple {i} ---")
    print("No stopwords :", df.loc[i, "no_stopwords"])
    print("Stemming     :", df.loc[i, "text_stem"])
    print("Lemmatisation:", df.loc[i, "text_lem"])


### Aperçu final du DataFrame

In [ ]:
display(df.head())
print("Colonnes finales :", list(df.columns))


# Partie I — Exercice 2 : Correction orthographique

On utilise ici une méthode **basée sur la distance de Levenshtein**. Pour chaque mot potentiellement mal orthographié, on cherche le mot du vocabulaire de référence qui minimise cette distance.

Le vocabulaire de référence est construit à partir de `corrPara`. Cette approche est volontairement simple et pédagogique : elle ne remplace pas un correcteur orthographique complet.


In [ ]:
para = """A beutiful and kand-heartedd gerrl nammed Ella and her parints livve hapily in a latge
hous withe a feiw sarvints, unttil her mather fals ille and laterr dies. Ella promise to folow her
mother's last wish: to have corage and be kind. Yaers latter, Ella's farher mrries recently-widowd
Ladi Tremaine, who has two unpleasent dauhters, Drisella and Anastasia. Ella's father leves on
busines, and Lady Tremaine rveals her cruel and jalous natur, forcing Ella to give up her bedrom
to the stepsisers and move into the atic. When Ella's father inexpectedly dies, Lady Tremaine
dismissses the houshold staff to save mony, and forcez all of theire chores upon Ella. Seeing Ella's
face covred in cinders after sleping by the firepplace, her step-family mokinglu dabs her
"Cinderella"."""

corrPara = """A beautiful and kind-hearted girl named Ella and her parents live happily in a large
house with a few servants, until her mother falls ill and later dies. Ella promises to follow her
mother's last wish: to have courage and be kind. Years later, Ella's father marries recently-widowed
Lady Tremaine, who has two unpleasant daughters, Drisella and Anastasia. Ella's father leaves on
business, and Lady Tremaine reveals her cruel and jealous nature, forcing Ella to give up her
bedroom to the stepsisters and move into the attic. When Ella's father unexpectedly dies, Lady
Tremaine dismisses the household staff to save money, and forces all of their chores upon Ella.
Seeing Ella's face covered in cinders after sleeping by the fireplace, her step-family mockingly
dubs her "Cinderella"."""

print("Paragraphe erroné :")
print(para)


### Fonctions de tokenisation et distance de Levenshtein

In [ ]:
def normalize_word(word):
    return re.sub(r"[^a-zA-ZÀ-ÿ'-]", "", word).lower()

def tokenize_words(text):
    return re.findall(r"[A-Za-zÀ-ÿ]+(?:[-'][A-Za-zÀ-ÿ]+)*", text)

def levenshtein_distance(a, b):
    # Programmation dynamique, sans dépendance externe.
    if len(a) < len(b):
        a, b = b, a

    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        current = [i]
        for j, cb in enumerate(b, start=1):
            insert = current[j - 1] + 1
            delete = previous[j] + 1
            substitute = previous[j - 1] + (ca != cb)
            current.append(min(insert, delete, substitute))
        previous = current
    return previous[-1]

ref_tokens = tokenize_words(corrPara)
ref_vocab = sorted(set(normalize_word(w) for w in ref_tokens if normalize_word(w)))

print("Nombre de mots distincts dans le vocabulaire de référence :", len(ref_vocab))
print(ref_vocab[:30])


### 1–2. Correction automatique et liste des mots corrigés

In [ ]:
# Seuil relatif : on ne corrige un mot que si le candidat est suffisamment proche.
# Les mots inconnus très courts sont laissés tels quels.
def correct_word(word, vocabulary, max_relative_distance=0.5):
    normalized = normalize_word(word)
    if not normalized or normalized in vocabulary:
        return word

    # Pour les mots très courts, éviter des corrections hasardeuses.
    if len(normalized) <= 2:
        return word

    best_word = None
    best_distance = float("inf")

    for candidate in vocabulary:
        # Petit filtre de longueur pour accélérer et éviter des correspondances absurdes.
        if abs(len(candidate) - len(normalized)) > max(2, len(normalized) // 2):
            continue
        d = levenshtein_distance(normalized, candidate)
        if d < best_distance:
            best_distance = d
            best_word = candidate

    if best_word is not None and best_distance <= max(1, int(len(normalized) * max_relative_distance)):
        return best_word

    return word

original_tokens = tokenize_words(para)
corrected_tokens = [
    correct_word(w, ref_vocab)
    for w in original_tokens
]

corrections = [(orig, corr) for orig, corr in zip(original_tokens, corrected_tokens)
               if normalize_word(orig) != normalize_word(corr)]

print("Mots corrigés (mot original -> correction) :")
for orig, corr in corrections:
    print(f"{orig:20s} -> {corr}")


### Reconstruction du paragraphe corrigé

In [ ]:
corrected_para = " ".join(corrected_tokens)

# Nettoyage léger des espaces avant certaines ponctuations.
corrected_para = re.sub(r"\s+([,.;:!?])", r"\1", corrected_para)
corrected_para = re.sub(r"\s+(["'])", r"\1", corrected_para)
corrected_para = re.sub(r"(["'])\s+", r"\1", corrected_para)

print(corrected_para)


### 3. Liste des mots incorrects selon le paragraphe de référence

In [ ]:
# Comparaison mot-à-mot avec normalisation.
reference_tokens = tokenize_words(corrPara)

n = min(len(original_tokens), len(reference_tokens))
incorrect_by_reference = []

for i in range(n):
    predicted = normalize_word(corrected_tokens[i])
    reference = normalize_word(reference_tokens[i])
    if predicted != reference:
        incorrect_by_reference.append({
            "position": i + 1,
            "mot_corrige": corrected_tokens[i],
            "mot_reference": reference_tokens[i]
        })

# Les éventuels mots supplémentaires/manquants sont également signalés.
if len(corrected_tokens) != len(reference_tokens):
    print(f"Attention : nombre de tokens corrigés={len(corrected_tokens)}, référence={len(reference_tokens)}")

incorrect_df = pd.DataFrame(incorrect_by_reference)
display(incorrect_df)


### 4. Taux d'erreurs par rapport à `corrPara`

In [ ]:
comparison_n = min(len(corrected_tokens), len(reference_tokens))
errors = sum(
    normalize_word(corrected_tokens[i]) != normalize_word(reference_tokens[i])
    for i in range(comparison_n)
)

# On ajoute les tokens en surplus comme erreurs.
errors += abs(len(corrected_tokens) - len(reference_tokens))

error_rate = errors / len(reference_tokens) if reference_tokens else 0

print("Nombre de tokens de référence :", len(reference_tokens))
print("Nombre d'erreurs              :", errors)
print(f"Taux d'erreurs                : {error_rate:.2%}")
print(f"Taux de correction             : {(1-error_rate):.2%}")


# Partie II — Exercice 1 : WordNet

### 1. Toutes les définitions possibles de `language`

In [ ]:
language_synsets = wn.synsets("language")

for i, synset in enumerate(language_synsets, start=1):
    print(f"{i}. {synset.name()}")
    print("   Définition :", synset.definition())
    print()


### 2. Exemple de phrase contenant `language`

In [ ]:
example_sentence = "Python is a programming language."
print(example_sentence)


### 3. Lemmes de `language` en anglais, italien, français et arabe

In [ ]:
languages = {
    "anglais": "eng",
    "italien": "ita",
    "français": "fra",
    "arabe": "arb",
}

for lang_name, lang_code in languages.items():
    lemmas = []
    for synset in wn.synsets("language", lang="eng"):
        for lemma in synset.lemmas(lang=lang_code):
            if lemma.name() not in lemmas:
                lemmas.append(lemma.name())
    print(f"{lang_name} ({lang_code}) :", lemmas)


### 4. Tous les synonymes de `language` et essai avec d'autres langues

In [ ]:
def synonyms(word, lang="eng"):
    result = set()
    for synset in wn.synsets(word, lang=lang):
        for lemma in synset.lemmas(lang=lang):
            result.add(lemma.name().replace("_", " "))
    return sorted(result)

for lang_code in ["eng", "ita", "fra", "arb"]:
    try:
        print(f"{lang_code} :", synonyms("language", lang_code))
    except Exception as e:
        print(f"{lang_code} : erreur -> {e}")


# Partie II — Exercice 2 : Corpus State of the Union

### 1. Vérification / installation des ressources

In [ ]:
# Les ressources ont déjà été téléchargées dans la cellule de préparation.
print("stopwords :", len(stopwords.words("english")), "mots")
print("Documents State of the Union :", state_union.fileids())


### 2. Charger le corpus et construire une liste de mots alphabétiques

In [ ]:
# Tous les mots du corpus.
raw_words = state_union.words()

# Uniquement les tokens constitués de lettres.
alpha_words = [word for word in raw_words if word.isalpha()]

print("Nombre total de tokens :", len(raw_words))
print("Nombre de mots alphabétiques :", len(alpha_words))
print("Premiers mots :", alpha_words[:30])


### 3. Supprimer les stopwords

In [ ]:
stop_words = set(stopwords.words("english"))

words_no_stopwords = [
    word for word in alpha_words
    if word.lower() not in stop_words
]

print("Avant suppression :", len(alpha_words))
print("Après suppression :", len(words_no_stopwords))
print("Premiers mots :", words_no_stopwords[:30])


### 4. Fréquences avec et sans `FreqDist`

In [ ]:
# Méthode 1 : collections.Counter (sans FreqDist)
counter_freq = collections.Counter(words_no_stopwords)

# Méthode 2 : FreqDist de NLTK
freq_dist = FreqDist(words_no_stopwords)

print("Top 20 avec Counter :")
print(counter_freq.most_common(20))

print("\nTop 20 avec FreqDist :")
print(freq_dist.most_common(20))


### Fréquence d'un même mot : minuscule, Première majuscule et CAPITALE

In [ ]:
# Exemple avec le mot "government" (présent sous plusieurs formes selon le corpus).
target = "government"

forms = [target.lower(), target.capitalize(), target.upper()]

for form in forms:
    print(f"{form:15s} -> {counter_freq[form]}")

print("\nRemarque : la fréquence est sensible à la casse.")
print("Government, government et GOVERNMENT sont comptés comme des chaînes différentes.")


### 5. Nouvelle distribution en normalisant les mots

In [ ]:
normalized_words = [word.lower() for word in words_no_stopwords]
normalized_freq = FreqDist(normalized_words)

print("Top 30 mots après normalisation :")
for word, freq in normalized_freq.most_common(30):
    print(f"{word:20s} {freq}")


### 6. Concordances du mot étudié sur le texte AVEC stopwords

In [ ]:
# On utilise la version avec les mots vides, comme demandé.
state_text = [word.lower() for word in raw_words if word.isalpha()]

# Choisir le mot de la question 5.
target_word = target.lower()

# Recherche manuelle des concordances : contexte de 7 mots de chaque côté.
window = 7
concordances = []

for i, word in enumerate(state_text):
    if word == target_word:
        left = state_text[max(0, i-window):i]
        right = state_text[i+1:i+1+window]
        concordances.append((" ".join(left), word, " ".join(right)))

print(f"Nombre de concordances pour '{target_word}' :", len(concordances))
for left, word, right in concordances[:20]:
    print(f"... {left} [{word}] {right} ...")


### 7. Collocations : bigrammes, trigrammes et quadgrammes

In [ ]:
# Les collocations sont calculées sur le texte normalisé et sans stopwords.
collocation_words = normalized_words

# Bigrammes
bigram_finder = BigramCollocationFinder.from_words(collocation_words)
bigram_scores = bigram_finder.score_ngrams(BigramAssocMeasures().likelihood_ratio)

print("Top 20 BIGRAMMES :")
for gram, score in bigram_scores[:20]:
    print(gram, "->", round(score, 4))

print("\nTop 20 TRIGRAMMES :")
trigram_finder = TrigramCollocationFinder.from_words(collocation_words)
trigram_scores = trigram_finder.score_ngrams(TrigramAssocMeasures().likelihood_ratio)
for gram, score in trigram_scores[:20]:
    print(gram, "->", round(score, 4))

print("\nTop 20 QUADGRAMMES :")
quadgram_finder = QuadgramCollocationFinder.from_words(collocation_words)
quadgram_scores = quadgram_finder.score_ngrams(QuadgramAssocMeasures().likelihood_ratio)
for gram, score in quadgram_scores[:20]:
    print(gram, "->", round(score, 4))


## Résumé des principales étapes réalisées

- Chargement et exploration du dataset Spam/Ham.
- Suppression de la ponctuation.
- Passage en minuscules.
- Tokenisation avec `re`.
- Suppression des stopwords NLTK.
- Stemming et lemmatisation.
- Correction orthographique par distance de Levenshtein.
- Calcul du taux d'erreurs par rapport au paragraphe de référence.
- Exploration de WordNet en plusieurs langues.
- Analyse du corpus *State of the Union*.
- Comptage des fréquences avec `Counter` et `FreqDist`.
- Normalisation des mots.
- Extraction de concordances.
- Extraction de bigrammes, trigrammes et quadgrammes.
